# 01 — FastAPI Core Concepts

ASGI vs WSGI, path operations, path/query/body parameters, automatic docs, and `TestClient` — the tool that lets every example in these notebooks run and be verified **inside a notebook**, with no live server or separate terminal needed.

## 0. Why `TestClient` instead of `uvicorn.run()`

A FastAPI app is normally served by an ASGI server (`uvicorn`) as a long-running process — awkward to "run" cell-by-cell in a notebook. `fastapi.testclient.TestClient` wraps the app and lets you call `client.get(...)`/`client.post(...)` **synchronously, in-process**, exercising the full routing/validation/dependency stack exactly as a real HTTP call would, just without a socket. This is also how FastAPI apps are unit-tested in practice — not a notebook-only trick.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*httpx2.*")   # silence a harmless TestClient/httpx notice

from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="demo-api")

@app.get("/health")
def health():
    return {"status": "ok"}

client = TestClient(app)
response = client.get("/health")
print(response.status_code)
print(response.json())

200
{'status': 'ok'}


## 1. ASGI vs WSGI — say this if asked

- **WSGI** (Flask, Django classic) — synchronous, one thread blocks per request; concurrency comes from running many worker processes/threads.
- **ASGI** (FastAPI, Starlette) — supports `async def` handlers, so a single worker can juggle many concurrent I/O-bound requests (waiting on a DB/HTTP call) without blocking a whole thread per request — plus native WebSocket/streaming support that WSGI can't express.

**FastAPI itself is built on Starlette** (ASGI toolkit) for the web parts, and **Pydantic** for data validation/serialization — this is a near-guaranteed "what is FastAPI built on" interview question.

## 2. Path parameters, query parameters, and type coercion

FastAPI infers where a parameter comes from: if its name appears in the route's `{...}` path template, it's a **path parameter**; otherwise (for simple types) it's a **query parameter**. Type hints aren't just documentation — FastAPI uses them to **validate and coerce** incoming strings (from the URL) into the declared Python type, returning a 422 automatically if that fails.

In [2]:
@app.get("/items/{item_id}")
def get_item(item_id: int, q: str | None = None, limit: int = 10):
    return {"item_id": item_id, "q": q, "limit": limit}

client = TestClient(app)

print(client.get("/items/42").json())
print(client.get("/items/42?q=search&limit=5").json())
print(client.get("/items/not-a-number").status_code)   # 422 -- int coercion fails

{'item_id': 42, 'q': None, 'limit': 10}
{'item_id': 42, 'q': 'search', 'limit': 5}
422


## 3. Request bodies with Pydantic models

Declaring a parameter's type as a Pydantic `BaseModel` tells FastAPI to parse the request body as JSON into that model — validating field types/constraints automatically, and rejecting malformed input with a structured 422 error before your function body even runs. This request/response validation is the core reason FastAPI is a common choice for data APIs — you get contract enforcement for free instead of hand-rolling `if "field" not in body` checks.

In [3]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float
    in_stock: bool = True

@app.post("/items")
def create_item(item: Item):
    return {"created": item.model_dump()}

client = TestClient(app)

ok = client.post("/items", json={"name": "widget", "price": 9.99})
print(ok.status_code, ok.json())

bad = client.post("/items", json={"name": "widget", "price": "not-a-number"})
print(bad.status_code)
print(bad.json())    # structured validation error, points at the exact field

200 {'created': {'name': 'widget', 'price': 9.99, 'in_stock': True}}
422
{'detail': [{'type': 'float_parsing', 'loc': ['body', 'price'], 'msg': 'Input should be a valid number, unable to parse string as a number', 'input': 'not-a-number'}]}


## 4. Status codes and response models

- Set the success status code with `status_code=` on the decorator (e.g. `201` for creation) rather than a magic number in the response body.
- `response_model=` declares the **output** shape and filters/validates the return value against it — useful for stripping internal-only fields (like a password hash) from what a DB-backed object actually returns.

In [4]:
from fastapi import status

class ItemOut(BaseModel):
    name: str
    price: float

class ItemInternal(BaseModel):
    name: str
    price: float
    cost_basis: float   # internal-only, should never reach the client

@app.post("/items-v2", response_model=ItemOut, status_code=status.HTTP_201_CREATED)
def create_item_v2(item: Item):
    internal = ItemInternal(name=item.name, price=item.price, cost_basis=item.price * 0.6)
    return internal    # response_model strips cost_basis before it's serialized

client = TestClient(app)
resp = client.post("/items-v2", json={"name": "widget", "price": 9.99})
print(resp.status_code)
print(resp.json())    # no cost_basis in the output

201
{'name': 'widget', 'price': 9.99}


## 5. Automatic interactive docs

FastAPI generates an OpenAPI schema from your route signatures and Pydantic models automatically, served at `/openapi.json`, with interactive UIs at `/docs` (Swagger UI) and `/redoc` — zero extra code required. This is one of FastAPI's headline selling points versus Flask, where you'd add a separate library (e.g. flask-smorest) for the same thing.

In [5]:
client = TestClient(app)
schema = client.get("/openapi.json").json()
print(list(schema["paths"].keys()))

['/health', '/items/{item_id}', '/items', '/items-v2']


## 6. Interview Q&A

1. **"What is FastAPI built on?"** — Starlette (ASGI web toolkit) for routing/requests/responses, Pydantic for data validation and serialization.
2. **"How does FastAPI decide if a parameter is a path param, query param, or body?"** — path params come from names matching `{...}` in the route; simple types not in the path default to query params; a `BaseModel`-typed parameter is parsed from the JSON body.
3. **"What does `response_model` actually do?"** — filters and validates the returned object against that schema before serializing, so extra/internal fields on the object you return never leak to the client.
4. **"Why ASGI over WSGI for a service that calls other APIs/DBs?"** — `async def` handlers let one worker handle many concurrent I/O-bound requests without a thread blocked per request, which matters most when the handler spends most of its time waiting on network I/O rather than doing CPU work.

## Summary

- `TestClient` exercises the full app in-process — the same tool used for both interview-notebook demos and real unit tests.
- Type hints double as validation/coercion rules for path/query params; `BaseModel` parameters validate JSON request bodies.
- `response_model` shapes and filters the output independently of what your handler actually returns.
- Next: `02_pydantic_validation_and_serialization.ipynb`.